# Exploratory Data Analysis — Olist E-Commerce

This notebook performs a comprehensive exploratory data analysis on the **Olist Brazilian E-Commerce** dataset.  
We examine key performance indicators, revenue trends, product categories, delivery performance,
customer reviews, and payment methods to surface actionable insights for the EcomIQ project.

In [ ]:
import sys
from pathlib import Path

# Add project root so src modules are importable
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Dark-theme colour palette
PAPER_BG  = '#0D1117'
PLOT_BG   = '#0D1117'
FONT_CLR  = '#94A3B8'
ACCENT    = '#00D4B1'

PLOTLY_LAYOUT = dict(
    paper_bgcolor=PAPER_BG,
    plot_bgcolor=PLOT_BG,
    font=dict(color=FONT_CLR),
)

print('Imports OK ✓')

In [ ]:
from src.data_loader import load_all_datasets, merge_master_dataframe

datasets = load_all_datasets(data_dir=PROJECT_ROOT / 'data' / 'raw')
df = merge_master_dataframe(datasets)
print(f'Master dataframe shape: {df.shape}')
df.head()

In [ ]:
print('=== DataFrame Info ===')
df.info()
print()
print(f'Shape: {df.shape}')
print()
print('=== Descriptive Statistics ===')
df.describe()

## KPI Summary

High-level business metrics derived from the merged dataset.

In [ ]:
total_orders     = df['order_id'].nunique()
total_revenue    = df['payment_value'].sum()
avg_order_value  = df.groupby('order_id')['payment_value'].sum().mean()
total_customers  = df['customer_unique_id'].nunique()

print(f'Total Orders       : {total_orders:>10,}')
print(f'Total Revenue (R$) : {total_revenue:>13,.2f}')
print(f'Avg Order Value    : {avg_order_value:>13,.2f}')
print(f'Total Customers    : {total_customers:>10,}')

## Monthly Revenue Trend

In [ ]:
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_month'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)

monthly_rev = (
    df.groupby('order_month')['payment_value']
    .sum()
    .reset_index()
    .rename(columns={'payment_value': 'revenue'})
)

fig = px.line(
    monthly_rev, x='order_month', y='revenue',
    title='Monthly Revenue Trend (R$)',
    labels={'order_month': 'Month', 'revenue': 'Revenue (R$)'},
)
fig.update_traces(line=dict(color=ACCENT, width=2.5))
fig.update_layout(**PLOTLY_LAYOUT)
fig.update_xaxes(tickangle=-45)
fig.show()

## Top 10 Product Categories by Revenue

In [ ]:
cat_rev = (
    df.groupby('product_category_name_english')['payment_value']
    .sum()
    .nlargest(10)
    .sort_values()
    .reset_index()
    .rename(columns={'product_category_name_english': 'category',
                     'payment_value': 'revenue'})
)

fig = px.bar(
    cat_rev, x='revenue', y='category', orientation='h',
    title='Top 10 Product Categories by Revenue',
    labels={'revenue': 'Revenue (R$)', 'category': 'Product Category'},
    color_discrete_sequence=[ACCENT],
)
fig.update_layout(**PLOTLY_LAYOUT)
fig.show()

## Order Status Distribution

In [ ]:
status_counts = df['order_status'].value_counts().reset_index()
status_counts.columns = ['status', 'count']

fig = px.pie(
    status_counts, names='status', values='count',
    title='Order Status Distribution',
    color_discrete_sequence=px.colors.sequential.Tealgrn,
)
fig.update_layout(**PLOTLY_LAYOUT)
fig.show()

## Customer State Distribution

In [ ]:
state_counts = (
    df.groupby('customer_state')['customer_unique_id']
    .nunique()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'customer_unique_id': 'customers', 'customer_state': 'state'})
)

fig = px.bar(
    state_counts, x='state', y='customers',
    title='Customer Distribution by State',
    labels={'state': 'State', 'customers': 'Unique Customers'},
    color_discrete_sequence=[ACCENT],
)
fig.update_layout(**PLOTLY_LAYOUT)
fig.show()

## Delivery Performance

Compare average actual delivery time versus estimated delivery time.

In [ ]:
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])
df['order_estimated_delivery_date'] = pd.to_datetime(df['order_estimated_delivery_date'])

delivered = df.dropna(subset=['order_delivered_customer_date']).copy()
delivered['actual_days'] = (
    (delivered['order_delivered_customer_date'] - delivered['order_purchase_timestamp'])
    .dt.days
)
delivered['estimated_days'] = (
    (delivered['order_estimated_delivery_date'] - delivered['order_purchase_timestamp'])
    .dt.days
)

avg_actual    = delivered['actual_days'].mean()
avg_estimated = delivered['estimated_days'].mean()

delivery_df = pd.DataFrame({
    'metric': ['Actual Delivery', 'Estimated Delivery'],
    'avg_days': [avg_actual, avg_estimated],
})

fig = px.bar(
    delivery_df, x='metric', y='avg_days',
    title='Average Delivery Time: Actual vs Estimated',
    labels={'metric': '', 'avg_days': 'Average Days'},
    color='metric',
    color_discrete_sequence=[ACCENT, '#6366F1'],
)
fig.update_layout(**PLOTLY_LAYOUT, showlegend=False)
fig.show()
print(f'Avg Actual Delivery   : {avg_actual:.1f} days')
print(f'Avg Estimated Delivery: {avg_estimated:.1f} days')

## Review Score Distribution

In [ ]:
review_counts = (
    df['review_score']
    .value_counts()
    .sort_index()
    .reset_index()
)
review_counts.columns = ['score', 'count']
review_counts['score'] = review_counts['score'].astype(str)

fig = px.bar(
    review_counts, x='score', y='count',
    title='Review Score Distribution (1–5)',
    labels={'score': 'Review Score', 'count': 'Number of Reviews'},
    color_discrete_sequence=[ACCENT],
)
fig.update_layout(**PLOTLY_LAYOUT)
fig.show()

## Payment Method Breakdown

In [ ]:
payment_counts = df['payment_type'].value_counts().reset_index()
payment_counts.columns = ['payment_type', 'count']

fig = px.pie(
    payment_counts, names='payment_type', values='count',
    title='Payment Method Breakdown',
    color_discrete_sequence=px.colors.sequential.Tealgrn,
    hole=0.35,
)
fig.update_layout(**PLOTLY_LAYOUT)
fig.show()

# Also show as a bar chart
fig2 = px.bar(
    payment_counts, x='payment_type', y='count',
    title='Payment Method Counts',
    labels={'payment_type': 'Payment Type', 'count': 'Count'},
    color_discrete_sequence=[ACCENT],
)
fig2.update_layout(**PLOTLY_LAYOUT)
fig2.show()

## Key Findings

1. **Revenue Growth** — Monthly revenue shows a clear upward trajectory with seasonal dips.
2. **Top Categories** — A handful of product categories drive the majority of revenue (e.g. bed/bath/table, health/beauty, electronics).
3. **Order Fulfilment** — The vast majority of orders are delivered; cancellation and unavailability rates are low.
4. **Geographic Concentration** — São Paulo (SP) dominates both customer count and order volume.
5. **Delivery Performance** — On average, orders are delivered faster than the estimated date, indicating conservative estimates.
6. **Review Sentiment** — Reviews skew positive (score 5 is most common), but a non-trivial cluster at score 1 warrants investigation.
7. **Payment Preferences** — Credit card is the overwhelming payment method of choice, followed by boleto.

---
*Next → `02_feature_engineering.ipynb` — Build RFM and behavioural features for churn modelling.*